# Phase 6 — Monitoring, Drift Detection & Governance

**Goal:** Ensure long-term reliability, safety, and compliance of the AI system.

## Contents
1. Data Validation Checks
2. Feature Drift Detection (PSI)
3. Prediction Distribution Monitoring
4. Audit Log Review
5. Governance Summary

In [ ]:
import pandas as pd, numpy as np, json, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns, warnings
warnings.filterwarnings('ignore')
import os
os.chdir('..') if 'notebooks' in os.getcwd() else None

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi':110, 'figure.figsize':(10,5)})

df = pd.read_csv('phase3_models/model_table.csv', parse_dates=['visit_date'])
df_sorted = df.sort_values('visit_date').reset_index(drop=True)
split_idx = int(len(df_sorted)*0.80)
train_df = df_sorted.iloc[:split_idx]
test_df  = df_sorted.iloc[split_idx:]
print(f'Train: {len(train_df):,}  Test: {len(test_df):,}')

## 1. Data Validation Checks

Input validation runs at the API boundary before any record reaches the model.

In [ ]:
with open('phase6_monitoring/validation_rules.json') as f:
    rules = json.load(f)

def validate_record(record: dict, rules: dict) -> list:
    """Returns list of validation errors (empty = valid)."""
    errors = []
    for field, rule in rules['input_validation_rules'].items():
        val = record.get(field)
        if val is None:
            errors.append(f'{field}: missing')
            continue
        if rule.get('min') is not None and val < rule['min']:
            errors.append(f'{field}: {val} < min {rule["min"]}')
        if rule.get('max') is not None and val > rule['max']:
            errors.append(f'{field}: {val} > max {rule["max"]}')
        if rule.get('allowed') and val not in rule['allowed']:
            errors.append(f'{field}: "{val}" not in {rule["allowed"]}')
    return errors

# Test valid record
valid = {'age':45,'length_of_stay_hours':12.0,'billed_amount':15000,'chronic_flag':0,
         'gender':'F','visit_type':'OPD','visit_month':6}
print('Valid record errors:  ', validate_record(valid, rules) or 'None ✅')

# Test invalid record
invalid = {'age':200,'length_of_stay_hours':-5,'billed_amount':-1000,'chronic_flag':3,
           'gender':'X','visit_type':'Walk-in','visit_month':15}
print('Invalid record errors:', validate_record(invalid, rules))

## 2. Feature Drift Detection (PSI)

**PSI (Population Stability Index)** measures how much a feature's distribution has shifted between training and production.

In [ ]:
drift_df = pd.read_csv('phase6_monitoring/drift_summary.csv')

fig, ax = plt.subplots(figsize=(10,5))
colors = ['#e74c3c' if f=='YES' else ('#f39c12' if f=='WATCH' else '#2ecc71')
          for f in drift_df['drift_flag']]
bars = ax.barh(drift_df['feature'], drift_df['psi'], color=colors)
ax.axvline(0.10, color='orange', linestyle='--', label='Watch (0.10)')
ax.axvline(0.20, color='red',    linestyle='--', label='Retrain (0.20)')
ax.set_title('Feature Drift — PSI Scores (Train vs Test)')
ax.set_xlabel('PSI'); ax.legend()
plt.tight_layout(); plt.savefig('phase6_monitoring/drift_chart.png', dpi=110); plt.show()

print('\nDrift Summary:')
display(drift_df[['feature','psi','drift_flag','mean_shift_pct']])

## 3. Prediction Distribution Monitoring

Monitor week-over-week distribution of predicted classes to detect concept drift.

In [ ]:
pred_log = pd.read_csv('phase6_monitoring/prediction_log_sample.csv')

risk_preds   = pred_log[pred_log['endpoint']=='/predict/risk']['prediction'].value_counts(normalize=True)*100
claim_preds  = pred_log[pred_log['endpoint']=='/predict/claim']['prediction'].value_counts(normalize=True)*100

fig, axes = plt.subplots(1,2,figsize=(12,4))
risk_preds.plot(kind='bar',  ax=axes[0], color=['#3498db','#e67e22','#e74c3c'])
axes[0].set_title('Predicted Risk Distribution (Sample)')
axes[0].set_ylabel('%'); axes[0].tick_params(rotation=0)

claim_preds.plot(kind='bar', ax=axes[1], color=['#2ecc71','#e74c3c','#f39c12'])
axes[1].set_title('Predicted Claim Distribution (Sample)')
axes[1].set_ylabel('%'); axes[1].tick_params(rotation=0)
plt.tight_layout(); plt.savefig('phase6_monitoring/prediction_distribution.png', dpi=110); plt.show()
display(pred_log)

## 4. Retraining Decision Framework

| Trigger | Condition | Action |
|---|---|---|
| Scheduled | Quarterly | Retrain on 12-month rolling window |
| Feature drift | PSI > 0.20 | Investigate + retrain within 30 days |
| Performance drop | High-risk recall < 0.65 | Immediate retrain |
| Business event | New insurer / new dept | Incremental retrain |

## 5. Governance Summary

In [ ]:
with open('phase4_evaluation/model_card.json') as f:
    model_card = json.load(f)

print('MODEL CARD')
print('='*60)
for model_key in ['model_a','model_b']:
    m = model_card[model_key]
    print(f"\n  {m['name']}")
    print(f"    Algorithm  : {m['algorithm']}")
    print(f"    Target     : {m['target']}")
    print(f"    Classes    : {m['classes']}")
    print(f"    Best Params: {m['best_params']}")
    print(f"    CV F1      : {m['cv_f1']}")
    print(f"    Test Acc   : {m['test_accuracy']}")
    print(f"    Key Metric : {m['primary_metric']}")

print(f"\n  Retraining: {model_card['retraining_trigger']}")
print(f"  Governance: {model_card['governance']}")
print('='*60)
print('\n✅ Phase 6 Complete')